# Project name (FALTA)
------
group ect(FLATA)

----
# Exploratory Data Analysis (EDA) 
## Summary

(FALTA)

## Index

(FALTA)


## Imports

In [118]:
#imports 
#(create utils file for functions)
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
# Resto de imports (visualization, etc)

In [119]:
df_AIAI_Customers_Complete = pd.read_csv('../../data/DM_AIAI_CustomerDB.csv')
df_AIAI_Customers = df_AIAI_Customers_Complete.copy()

df_AIAI_Flights_Complete = pd.read_csv('../../data/DM_AIAI_FlightsDB.csv')
df_AIAI_Flights = df_AIAI_Flights_Complete.copy()

## Metadata

DM_AIAI_CustomerDB.csv

- *Loyalty#* - Unique customer identifier for loyalty program members
- *First Name* - Customer’s first name
- *Last Name* - Customer’s last name
- *Customer Name* - Customer’s full name (concatenated)
- *Country* - Customer’s country of residence
- *Province or State* - Customer’s province or state
- *City* - Customer’s city of residence
- *Latitude* - Geographic latitude coordinate of customer location
- *Longitude* - Geographic longitude coordinate of customer location
- *Postal code* - Customer’s postal/ZIP code
- *Gender* - Customer’s gender
- *Education* - Customer’s highest education level (Bachelor, College, etc.)
- *Location Code* - Urban/Suburban/Rural classification of customer residence
- *Income* - Customer’s annual income
- *Marital Status* - Customer’s marital status (Married, Single, Divorced)
- *LoyaltyStatus* - Current tier status in loyalty program (Star > Nova > Aurora)
- *EnrollmentDateOpening* - Date when customer joined the loyalty program
- *CancellationDate* - Date when customer left the program
- *Customer Lifetime Value* - Total calculated monetary value of customer relationship
- *EnrollmentType* - Method of joining loyalty program

DM_AIAI_FlightsDB.csv


- *Loyalty#* - Unique customer identifier linking to CustomerDB
- *Year* - Year of flight activity record
- *Month* - Month of flight activity record (1-12)
- *YearMonthDate* - First day of the month for the activity period
- *NumFlights* -  Total number of flights taken by customer in the month
- *NumFlightsWithCompanions* - Number of flights where customer traveled with companions
- *DistanceKM* - Total distance traveled in kilometers for the month
- *PointsAccumulated* - Loyalty points earned by customer during the month
- *PointsRedeemed* - Loyalty points spent/redeemed by customer during the month
- *DollarCostPointsRedeemed* - Dollar value of points redeemed during the month


## Data Exploration

#### **Customers DataFrame**

In [120]:
df_AIAI_Customers.head()

,Unnamed: 0,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,...,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
0,0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,...,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
1,1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,...,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
2,2,429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,...,male,College,Urban,0.0,Single,Star,7/14/2017,1/8/2021,3839.75,Standard
3,3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,...,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
4,4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,...,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion


Delete the first column that is not an index nor in the metadata

In [121]:
df_AIAI_Customers.drop(columns=['Unnamed: 0'], inplace=True)

Now we will verify the different types of data and the missing values

In [122]:
df_AIAI_Customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16921 entries, 0 to 16920
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 16921 non-null  int64  
 1   First Name               16921 non-null  object 
 2   Last Name                16921 non-null  object 
 3   Customer Name            16921 non-null  object 
 4   Country                  16921 non-null  object 
 5   Province or State        16921 non-null  object 
 6   City                     16921 non-null  object 
 7   Latitude                 16921 non-null  float64
 8   Longitude                16921 non-null  float64
 9   Postal code              16921 non-null  object 
 10  Gender                   16921 non-null  object 
 11  Education                16921 non-null  object 
 12  Location Code            16921 non-null  object 
 13  Income                   16901 non-null  float64
 14  Marital Status        

We have on total 16921 observations (customers).

The data types are correct on almost every column, meaning that the values stores at least are of the data types that they are supouse to be.

It is only needed to change the data types of the date columns (EnrollmentDateOpening, CancellationDate).

In [123]:
df_AIAI_Customers.isna().sum()

Loyalty#                       0
First Name                     0
Last Name                      0
Customer Name                  0
Country                        0
Province or State              0
City                           0
Latitude                       0
Longitude                      0
Postal code                    0
Gender                         0
Education                      0
Location Code                  0
Income                        20
Marital Status                 0
LoyaltyStatus                  0
EnrollmentDateOpening          0
CancellationDate           14611
Customer Lifetime Value       20
EnrollmentType                 0
dtype: int64

We have missing values in our data in the columns: Income (20 records, 0.11%), CancellationDate (14611 records, 86%), Customer Lifetime Value (20 records, 0.11%). We will evaluate and them decide if we can remove those observations (since they are a small proportion of the hole dataset) or do imputation.

Probably people do not fill the income because they do not want us to know how much they earn, so we cuold perform an imputation with similar data points.

The higher number of missing values in CancellationDate may be because not many people canceled and didn't leave the program. So it is not an error.

To know what to do with Customer Lifetime Value, we will need to see the relation that exits with the flights data, if the customer have some records if better not to remove and instead do some imputation based on similar points (performing imputation).

In [124]:
# Identify rows with more than one missing value in 'Income' and 'Customer Lifetime Value'
df_AIAI_Customers[df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1]

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard
16906,999992,Ella,Roy,Ella Roy,Canada,Ontario,Toronto,43.706878,-79.437412,P6D 6N2,male,College,Suburban,NaN,Single,Star,3/27/2021,3/27/2021,NaN,Standard
16907,999993,Elijah,Cook,Elijah Cook,Canada,British Columbia,Dawson Creek,55.701475,-120.181716,W6H 0Z7,female,College,Suburban,NaN,Married,Star,1/27/2015,1/27/2015,NaN,Standard
16908,999994,Ethan,Chan,Ethan Chan,Canada,Ontario,Ottawa,45.365906,-75.723181,B2F 3E1,female,College,Rural,NaN,Married,Star,5/5/2016,5/5/2016,NaN,Standard
16909,999995,Liam,Wong,Liam Wong,Canada,Ontario,Ottawa,45.471557,-75.704868,B3A 2R0,female,College,Suburban,NaN,Married,Star,3/2/2020,3/2/2020,NaN,Standard
16910,999996,Isabella,Ross,Isabella Ross,Canada,Ontario,Toronto,43.690489,-79.436758,B4W 4M6,female,Bachelor,Suburban,NaN,Single,Star,9/14/2018,9/14/2018,NaN,Standard


In [125]:
# See if all 20 rows with missing are misisng on both values in 'Income' and 'Customer Lifetime Value' 
df_AIAI_Customers[df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1].info()
# Awnser: Yes

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 16901 to 16920
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 20 non-null     int64  
 1   First Name               20 non-null     object 
 2   Last Name                20 non-null     object 
 3   Customer Name            20 non-null     object 
 4   Country                  20 non-null     object 
 5   Province or State        20 non-null     object 
 6   City                     20 non-null     object 
 7   Latitude                 20 non-null     float64
 8   Longitude                20 non-null     float64
 9   Postal code              20 non-null     object 
 10  Gender                   20 non-null     object 
 11  Education                20 non-null     object 
 12  Location Code            20 non-null     object 
 13  Income                   0 non-null      float64
 14  Marital Status           2

In [126]:
# Identify rows when 'EnrollmentDateOpening' is the same as 'CancellationDate'
df_AIAI_Customers[df_AIAI_Customers["EnrollmentDateOpening"] == df_AIAI_Customers["CancellationDate"] ]

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
3153,488724,Rayford,Vogus,Rayford Vogus,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,male,Bachelor,Suburban,36495.0,Married,Aurora,1/5/2016,1/5/2016,10629.22,Standard
12237,871455,Gemma,Gadbois,Gemma Gadbois,Canada,Alberta,Calgary,51.048615,-114.070850,T3E 2V9,male,Doctor,Suburban,16269.0,Divorced,Star,6/18/2020,6/18/2020,3211.07,Standard
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard
16906,999992,Ella,Roy,Ella Roy,Canada,Ontario,Toronto,43.706878,-79.437412,P6D 6N2,male,College,Suburban,NaN,Single,Star,3/27/2021,3/27/2021,NaN,Standard
16907,999993,Elijah,Cook,Elijah Cook,Canada,British Columbia,Dawson Creek,55.701475,-120.181716,W6H 0Z7,female,College,Suburban,NaN,Married,Star,1/27/2015,1/27/2015,NaN,Standard
16908,999994,Ethan,Chan,Ethan Chan,Canada,Ontario,Ottawa,45.365906,-75.723181,B2F 3E1,female,College,Rural,NaN,Married,Star,5/5/2016,5/5/2016,NaN,Standard


When the customer has the missing on the income variable, it also do not have Lifetime Value, and they canceled the suscription on the same day, meaning these are persons that are not really customers. At this point the aproach of remove those observations is the better.

Is important to say that are 2 customers that enrolled and canceled on the same day but they have some monetary value, meaning that they only purchased on 1 day, but they are customers (loyalty#: 488724 and 871455).

In [127]:
df_AIAI_Flights[df_AIAI_Flights["Loyalty#"].isin([488724,871455])].describe(include="all")

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
count,72.000000,72.000000,72.000000,72,72.0,72.0,72.0,72.0,72.0,72.0
unique,NaN,NaN,NaN,36,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,6/1/2020,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
mean,680089.500000,2020.000000,6.500000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
std,192708.432265,0.822226,3.476278,NaN,0.0,0.0,0.0,0.0,0.0,0.0
min,488724.000000,2019.000000,1.000000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
25%,488724.000000,2019.000000,3.750000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
50%,680089.500000,2020.000000,6.500000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
75%,871455.000000,2021.000000,9.250000,NaN,0.0,0.0,0.0,0.0,0.0,0.0


We will also drop these two customers since they dont have any real records of flights (no variance on their transactions)

**Duplicates**

In [128]:
df_AIAI_Customers.duplicated().sum()
# No duplicate rows found

np.int64(0)

In [129]:
df_AIAI_Customers.duplicated(subset=['Loyalty#'], keep=False).sum()
# 327 duplicate Loyalty# values found

np.int64(327)

In [130]:
duplicates_loyalty_Customers_mask = df_AIAI_Customers.duplicated(subset=['Loyalty#'], keep=False)
df_AIAI_Customers[duplicates_loyalty_Customers_mask].sort_values(['Loyalty#'])

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
1646,101902,Hans,Schlottmann,Hans Schlottmann,Canada,Ontario,London,42.984924,-81.245277,M5B 3E4,female,College,Rural,0.0,Married,Aurora,1/7/2020,NaN,6265.34,Standard
2668,101902,Yi,Nesti,Yi Nesti,Canada,Ontario,Toronto,43.653225,-79.383186,M8Y 4K8,female,Bachelor,Urban,79090.0,Married,Aurora,3/19/2020,NaN,8609.16,Standard
15988,106001,Maudie,Hyland,Maudie Hyland,Canada,New Brunswick,Fredericton,45.963589,-66.643112,E3B 2H2,female,Master,Suburban,14973.0,Divorced,Star,7/16/2015,NaN,12168.74,Standard
700,106001,Ivette,Peifer,Ivette Peifer,Canada,Quebec,Montreal,45.501690,-73.567253,H2Y 4R4,female,High School or Below,Suburban,10037.0,Single,Star,1/11/2016,NaN,4914.04,Standard
13053,106509,Stacy,Schwebke,Stacy Schwebke,Canada,Ontario,Toronto,43.653225,-79.383186,P1J 8T7,female,College,Suburban,0.0,Single,Star,6/12/2021,NaN,4661.98,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,989528,Sharri,Boughman,Sharri Boughman,Canada,Quebec,Montreal,45.501690,-73.567253,H2T 2J6,female,College,Rural,0.0,Divorced,Nova,5/1/2020,NaN,3370.07,Standard
9890,990512,Magda,Sopher,Magda Sopher,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,10/21/2018,NaN,1904.00,Standard
14478,990512,Ione,Snowden,Ione Snowden,Canada,British Columbia,Vancouver,49.282730,-123.120740,V5R 1W3,female,College,Urban,0.0,Single,Star,8/20/2021,NaN,6870.61,Standard
16380,992168,Crysta,Bennin,Crysta Bennin,Canada,Ontario,Ottawa,45.421532,-75.697189,K1F 2R2,female,Master,Urban,22828.0,Married,Star,12/18/2017,NaN,16473.17,Standard


We will drop these records since there are people with the same Loyalty number and different characteristics, and we do not know which is the "real person", and since we need to use that loyalty number to identify the customer in the other dataset we will drop them (are only 327 rows out of 16921, so it is only 1.93% of the dataset).

And we can put as index colunm the Loyalty number since now we know that will be unique.

**Inconsistencies**

In [131]:
df_AIAI_Customers[df_AIAI_Customers['CancellationDate'] == "2/29/2019"]
# We found that there is a wrong date format, since that day doesnt exist, we will change it to 2/28/2019

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
8840,314558,Retta,Pauley,Retta Pauley,Canada,Quebec,Montreal,45.501690,-73.567253,H4G 3T4,male,Bachelor,Rural,54311.0,Married,Nova,9/5/2017,2/29/2019,10722.06,Standard
14757,373118,Quinn,Shamonsky,Quinn Shamonsky,Canada,Manitoba,Winnipeg,49.895138,-97.138374,R3R 3T4,male,College,Rural,0.0,Single,Star,6/29/2018,2/29/2019,7516.79,Standard


In [132]:
df_AIAI_Customers.loc[df_AIAI_Customers['CancellationDate'] == "2/29/2019", 'CancellationDate'] = "2/28/2019"

**Fix dataset on previous analysis**

In [133]:
# Removing duplicate Loyalty# rows
df_AIAI_Customers = df_AIAI_Customers[~(duplicates_loyalty_Customers_mask)]

# Removing rows with missing Income and Customer Lifetime Value
df_AIAI_Customers = df_AIAI_Customers[~(df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1)]

# Removing the two customers that enrolled and canceled on the same day
df_AIAI_Customers = df_AIAI_Customers[~(df_AIAI_Customers["Loyalty#"].isin([488724,871455]))]

# Setting Loyalty# as index
df_AIAI_Customers.set_index('Loyalty#', inplace=True)

# Putting the date columns on datetime format
df_AIAI_Customers['EnrollmentDateOpening'] = pd.to_datetime(df_AIAI_Customers['EnrollmentDateOpening'], format='%m/%d/%Y')
df_AIAI_Customers['CancellationDate'] = pd.to_datetime(df_AIAI_Customers['CancellationDate'], format='%m/%d/%Y')

In [134]:
df_AIAI_Customers.head()

,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
Loyalty#,,,,,,,,,,,,,,,,,,,
480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2019-02-15,NaT,3839.14,Standard
549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,2019-03-09,NaT,3839.61,Standard
429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,College,Urban,0.0,Single,Star,2017-07-14,2021-01-08,3839.75,Standard
608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2016-02-17,NaT,3839.75,Standard
530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,2017-10-25,NaT,3842.79,2021 Promotion


In [135]:
df_AIAI_Customers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16572 entries, 480934 to 652627
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   First Name               16572 non-null  object        
 1   Last Name                16572 non-null  object        
 2   Customer Name            16572 non-null  object        
 3   Country                  16572 non-null  object        
 4   Province or State        16572 non-null  object        
 5   City                     16572 non-null  object        
 6   Latitude                 16572 non-null  float64       
 7   Longitude                16572 non-null  float64       
 8   Postal code              16572 non-null  object        
 9   Gender                   16572 non-null  object        
 10  Education                16572 non-null  object        
 11  Location Code            16572 non-null  object        
 12  Income                   16572 

In [136]:
df_AIAI_Customers.isna().sum()

First Name                     0
Last Name                      0
Customer Name                  0
Country                        0
Province or State              0
City                           0
Latitude                       0
Longitude                      0
Postal code                    0
Gender                         0
Education                      0
Location Code                  0
Income                         0
Marital Status                 0
LoyaltyStatus                  0
EnrollmentDateOpening          0
CancellationDate           14327
Customer Lifetime Value        0
EnrollmentType                 0
dtype: int64

Out of the 16921 observations that we had, we now have 16572, meaning that until now we removed 349 records, that is 2,06% of the hole dataset.

In [137]:
df_AIAI_Customers.describe()

,Latitude,Longitude,Income,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value
count,16572.000000,16572.000000,16572.000000,16572,2245,16572.000000
mean,47.174612,-91.839905,37741.357531,2018-10-07 04:29:43.055756800,2019-12-20 11:55:49.844098048,7986.623409
min,42.984924,-135.056840,0.000000,2015-04-01 00:00:00,2015-11-30 00:00:00,1898.010000
25%,44.231171,-120.237660,0.000000,2017-01-18 18:00:00,2019-02-02 00:00:00,3979.127500
50%,46.087818,-79.383186,34137.000000,2018-11-02 00:00:00,2020-01-14 00:00:00,5780.180000
75%,49.282730,-74.596184,62375.000000,2020-07-11 06:00:00,2021-02-15 00:00:00,8954.430000
max,60.721188,-52.712578,99981.000000,2021-12-30 00:00:00,2021-12-30 00:00:00,83325.380000
std,3.305572,22.240873,30357.279626,NaN,NaN,6858.782006


Conclusions:
+ We do not have any inconsistencies like negative income, lifetime customer value, or very old dates.
+ Having income with value equal to 0 is strage but we need some visualizations to conclude about that. 
+ We will see if there are still people with same enrollement and cancelation date.
+ Our oldest customer enrrolled on 01-04-2015, and our most recent customer enrolled on 30-12-2021.
+ We will check for people with older enrollement day than the cancellation date since these are customer that came back to our loyalty program after cancellation.

In [138]:
df_AIAI_Customers[df_AIAI_Customers["CancellationDate"] == df_AIAI_Customers["EnrollmentDateOpening"]]
# No rows with same Enrollment and Cancellation date

,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
Loyalty#,,,,,,,,,,,,,,,,,,,


In [139]:
# Re-engaged customers that were lost (canceled the Loyalty and came back)
df_AIAI_Customers[df_AIAI_Customers["EnrollmentDateOpening"] >= df_AIAI_Customers["CancellationDate"] ]

,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
Loyalty#,,,,,,,,,,,,,,,,,,,
357549,Elisha,Furna,Elisha Furna,Canada,British Columbia,Whistler,50.116322,-122.957360,V6T 1Y8,female,Bachelor,Rural,60817.0,Single,Star,2021-09-21,2019-05-21,3964.73,Standard
265297,Ebonie,Radde,Ebonie Radde,Canada,Manitoba,Winnipeg,49.895138,-97.138374,R2C 0M5,female,Bachelor,Urban,39101.0,Married,Star,2021-07-17,2019-03-17,3978.67,Standard
845613,Jerald,Shiring,Jerald Shiring,Canada,Quebec,Montreal,45.501690,-73.567253,H2Y 4R4,male,Bachelor,Urban,30598.0,Married,Star,2021-10-13,2019-06-13,4198.03,Standard
830547,Dortha,Detar,Dortha Detar,Canada,British Columbia,Vancouver,49.282730,-123.120740,V5R 1W3,male,Bachelor,Rural,26245.0,Single,Star,2021-05-11,2019-01-11,4271.20,Standard
514900,Edith,Forslin,Edith Forslin,Canada,Ontario,Toronto,43.653225,-79.383186,P1L 8X8,female,Bachelor,Urban,80892.0,Married,Star,2021-07-20,2019-03-20,4425.91,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584796,Julieann,Mclaughlan,Julieann Mclaughlan,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,Bachelor,Rural,59853.0,Single,Star,2021-08-18,2019-04-18,19454.41,Standard
774931,Cira,Defide,Cira Defide,Canada,British Columbia,Whistler,50.116322,-122.957360,V6T 1Y8,female,Bachelor,Rural,33588.0,Married,Star,2021-07-21,2019-03-21,19731.34,Standard
275807,Arturo,Michaeli,Arturo Michaeli,Canada,Manitoba,Winnipeg,49.895138,-97.138374,R2C 0M5,female,Bachelor,Urban,71467.0,Divorced,Star,2021-10-18,2019-06-18,20446.60,Standard


199 Customer had Re-engaged, we can create a new variable flaging these customers.

In [140]:
df_AIAI_Customers.describe(include="O")

,First Name,Last Name,Customer Name,Country,Province or State,City,Postal code,Gender,Education,Location Code,Marital Status,LoyaltyStatus,EnrollmentType
count,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572
unique,4926,15114,16572,1,11,29,55,2,5,3,3,3,2
top,Stacey,Salberg,Ariane Peyton,Canada,Ontario,Toronto,V6E 3D9,female,Bachelor,Suburban,Married,Star,Standard
freq,13,4,1,16572,5353,3322,906,8335,10377,5606,9645,7597,15434


Conclusions:
+ The information is only about people that are living on Canada, so que can remove that column.
+ We have 2 enrrolment type, being one more with way more records than the other.
+ Each person has a unique name.
+ We need to see the Postal code since there are only 55 different postal code, that is not normal and we may delete also that variable (will depend on the conclusions of the geographical analysis). 

In [141]:
df_AIAI_Customers.drop(columns=['Country'], inplace=True)

#### **Flights DataFrame**

In [142]:
df_AIAI_Flights.head()

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
0,413052,2021,12,12/1/2021,2.0,2.0,9384.0,938.0,0.0,0.0
1,464105,2021,12,12/1/2021,0.0,0.0,0.0,0.0,0.0,0.0
2,681785,2021,12,12/1/2021,10.0,3.0,14745.0,1474.0,0.0,0.0
3,185013,2021,12,12/1/2021,16.0,4.0,26311.0,2631.0,3213.0,32.0
4,216596,2021,12,12/1/2021,9.0,0.0,19275.0,1927.0,0.0,0.0


In [143]:
df_AIAI_Flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608436 entries, 0 to 608435
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Loyalty#                  608436 non-null  int64  
 1   Year                      608436 non-null  int64  
 2   Month                     608436 non-null  int64  
 3   YearMonthDate             608436 non-null  object 
 4   NumFlights                608436 non-null  float64
 5   NumFlightsWithCompanions  608436 non-null  float64
 6   DistanceKM                608436 non-null  float64
 7   PointsAccumulated         608436 non-null  float64
 8   PointsRedeemed            608436 non-null  float64
 9   DollarCostPointsRedeemed  608436 non-null  float64
dtypes: float64(6), int64(3), object(1)
memory usage: 46.4+ MB


In [144]:
df_AIAI_Flights.isna().sum()

Loyalty#                    0
Year                        0
Month                       0
YearMonthDate               0
NumFlights                  0
NumFlightsWithCompanions    0
DistanceKM                  0
PointsAccumulated           0
PointsRedeemed              0
DollarCostPointsRedeemed    0
dtype: int64

We have on total 608436 observations (records of flights), with 0 missing values.

The data types are correct on almost every column, meaning that the values stores at least are of the data types that they are supouse to be.

It is only needed to change the data types of the date column YearMonthDate, and transform the colunm NumFlights and NumFlightsWithCompanions to integers, since they should be discrete.

**Duplicates**

In [145]:
df_AIAI_Flights.duplicated(keep=False).sum()
# We have 5778 duplicate rows

np.int64(5778)

We will only keep 1 record per each duplicate since it has the same information.

In [146]:
df_AIAI_Flights = df_AIAI_Flights.drop_duplicates(keep='first')

**Inconsistencies**

We will check for records that do not have any information (0 flights, 0 distance, 0 points...), to have only information of actual travels.

In [147]:
(df_AIAI_Flights['NumFlights'] == 0).sum()
# Almost half of the records have 0 flights

np.int64(301621)

In [148]:
((df_AIAI_Flights['NumFlights'] == 0) & (df_AIAI_Flights['DistanceKM'] == 0)).sum()

np.int64(295720)

In [149]:
(df_AIAI_Flights['DistanceKM'] == 0).sum()
# When NumFlights is 0, DistanceKM is also 0

np.int64(295720)

Since there are 5901 flights with some distance but with 0 flights registered we will impute those zero values using only the DistanceKM, to not lose those records, the rest of values that has 0 flights and 0 distance will be removed, since they dond add any information. 

In [150]:
# Remove inconsistency about 0 flights and 0 distance
df_AIAI_Flights = df_AIAI_Flights[~((df_AIAI_Flights['NumFlights'] == 0) & (df_AIAI_Flights['DistanceKM'] == 0))]

In [151]:
# Fill the 5901 records with 0 flights but some distance using KNN imputer
mising_nflights_mask = df_AIAI_Flights['NumFlights'] == 0

# Prepare data for imputation
X = df_AIAI_Flights.loc[~mising_nflights_mask, ['DistanceKM']]  # Training data
y = df_AIAI_Flights.loc[~mising_nflights_mask, ['NumFlights']]   # Target values

# Initialize and fit KNN imputer
imputer = KNNImputer(n_neighbors=10)
imputer.fit(X, y)

# Predict values for rows needing imputation
X_missing = df_AIAI_Flights.loc[mising_nflights_mask, ['DistanceKM']]
imputed_values = imputer.transform(X_missing)

# Update original dataframe with imputed values
df_AIAI_Flights.loc[mising_nflights_mask, 'NumFlights'] = np.round(imputed_values.flatten())

(AQUI ME QUEDE) ver si hay repetidos/duplicados en loyalty, e fechas

In [116]:
df_AIAI_Flights[mising_nflights_mask] 

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
19,261109,2021,12,12/1/2021,13736.0,0.0,13736.0,1373.00,0.0,0.0
93,817609,2021,12,12/1/2021,23775.0,0.0,23775.0,2377.00,0.0,0.0
96,192600,2021,12,12/1/2021,5119.0,0.0,5119.0,511.00,0.0,0.0
116,883242,2021,12,12/1/2021,20681.0,0.0,20681.0,2068.00,0.0,0.0
154,493800,2021,12,12/1/2021,17502.0,0.0,17502.0,1750.00,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
607387,944702,2019,12,12/1/2019,29223.0,0.0,29223.0,2922.30,0.0,0.0
607549,952629,2019,12,12/1/2019,27343.0,0.0,27342.9,2734.29,0.0,0.0
607594,954973,2019,12,12/1/2019,6437.0,0.0,6436.8,643.68,0.0,0.0
607744,962989,2019,12,12/1/2019,15923.0,0.0,15922.8,1592.28,0.0,0.0


In [152]:
(df_AIAI_Flights['NumFlights'] == 0).sum()
# All the records now have some value on NumFlights

np.int64(0)

In [153]:
df_AIAI_Flights.describe()

,Loyalty#,Year,Month,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
count,309813.000000,309813.000000,309813.000000,309813.000000,309813.000000,309813.000000,309813.00000,309813.000000,309813.000000
mean,549016.626446,2020.049646,6.644686,308.193004,1.932350,15591.925245,1558.88545,462.006404,4.565700
std,258323.399102,0.825077,3.449062,2498.150124,2.460212,9350.482046,935.03301,1339.335181,13.248052
min,100018.000000,2019.000000,1.000000,0.900000,0.000000,218.700000,21.87000,0.000000,0.000000
25%,326699.000000,2019.000000,4.000000,4.000000,0.000000,7808.400000,780.84000,0.000000,0.000000
50%,549612.000000,2020.000000,7.000000,8.000000,0.900000,15069.000000,1506.60000,0.000000,0.000000
75%,771353.000000,2021.000000,10.000000,11.000000,3.600000,22563.000000,2256.00000,0.000000,0.000000
max,999986.000000,2021.000000,12.000000,40109.000000,11.000000,42040.000000,4204.00000,7496.000000,74.000000


(CHECK) 
total points redemmed more than points acumulated
number of flights with companions, and number of flights pass to integer


(NEW VARIABLES)
Days with us: (since enrollment of the program, until cancelation (if smaller), else until today)
Re-engaged people: (have a enrrolment after the canceled, flag those)
Days until re-engament: only the ones that are re-engaged.

Pass categorical variables (gender, education, Martial status(binary), Loyalty status) to dummies -1.

Number of flights:
Total Distance:
Mean distance per flight:
number of flights with companions:
% of flights with companions:
points acumulated:
points redeemed:
Dollar cost point redemmed (points/100):
Dollar cost points still for redeming (acummulating - redemed /100):

Mean flights per year:
mean distance per year:
mean points acumulated, redemed, dolar cost for redeming per year:

Flights per month:
Month with more flights:
Distance per month:
Month with more distance:
Points acumulated, redemed, still for redemen dollar cost:


Variables not use on cluster: names(all), Enrrolment type, citys and urbans, points redemmed and acummulated (use for redeming)